In [1]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
# Cell 2: Load & preprocess dataset
df = pd.read_csv("../Dataset/QK-video_subset_5M.csv")

# Target
TARGET = "click"

# Categorical & Numeric features
categorical_cols = ["user_id", "item_id", "video_category", "gender", "age"]
numeric_cols = ["watching_times"]

# Label encode categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Scale numeric features
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Train-test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df.shape, test_df.shape


((4000000, 10), (1000000, 10))

In [3]:
# Cell 3: Dataset class
class TenrecDataset(Dataset):
    def __init__(self, df, categorical_cols, numeric_cols, target_col):
        self.df = df
        self.categorical_data = df[categorical_cols].values
        self.numeric_data = df[numeric_cols].values
        self.targets = df[target_col].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.categorical_data[idx], dtype=torch.long),
            torch.tensor(self.numeric_data[idx], dtype=torch.float32),
            torch.tensor(self.targets[idx], dtype=torch.float32)
        )

train_dataset = TenrecDataset(train_df, categorical_cols, numeric_cols, TARGET)
test_dataset = TenrecDataset(test_df, categorical_cols, numeric_cols, TARGET)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [4]:
# Cell 4: Wide & Deep model
class WideAndDeep(nn.Module):
    def __init__(self, cat_dims, num_dim, embed_dim=8, hidden_dims=[64, 32]):
        super(WideAndDeep, self).__init__()
        
        # 🔵 Embedding cho Deep part
        self.embeddings = nn.ModuleList([nn.Embedding(cat_dim, embed_dim) for cat_dim in cat_dims])
        
        # 🟠 Wide part (linear với one-hot)
        self.wide = nn.Linear(sum(cat_dims) + num_dim, 1)  # one-hot của categorical + numeric
        
        # 🟣 Deep part (MLP)
        deep_input_dim = len(cat_dims) * embed_dim + num_dim
        layers = []
        for h in hidden_dims:
            layers.append(nn.Linear(deep_input_dim, h))
            layers.append(nn.ReLU())
            deep_input_dim = h
        layers.append(nn.Linear(deep_input_dim, 1))
        self.deep = nn.Sequential(*layers)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x_categorical, x_numeric):
        # Embedding cho deep part
        embed_out = [emb(x_categorical[:, i]) for i, emb in enumerate(self.embeddings)]
        embed_out = torch.cat(embed_out, dim=1)  # (batch_size, cat_num * embed_dim)
        
        # One-hot cho wide part
        one_hot_parts = []
        offset = 0
        for i, cat_dim in enumerate(cat_dims):
            one_hot = torch.nn.functional.one_hot(x_categorical[:, i], num_classes=cat_dim)
            one_hot_parts.append(one_hot)
            offset += cat_dim
        one_hot_cat = torch.cat(one_hot_parts, dim=1).float()
        
        wide_input = torch.cat([one_hot_cat, x_numeric], dim=1)
        deep_input = torch.cat([embed_out, x_numeric], dim=1)
        
        wide_out = self.wide(wide_input)
        deep_out = self.deep(deep_input)
        
        out = wide_out + deep_out
        return self.sigmoid(out)


In [5]:
# Cell 5: Setup training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cat_dims = [df[col].nunique() for col in categorical_cols]
num_dim = len(numeric_cols)

model = WideAndDeep(cat_dims=cat_dims, num_dim=num_dim).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
# Cell 6: Training & evaluation functions
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for x_cat, x_num, y in train_loader:
        x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(x_cat, x_num).squeeze()
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate_model(model, test_loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x_cat, x_num, y in test_loader:
            x_cat, x_num = x_cat.to(device), x_num.to(device)
            preds = model(x_cat, x_num).squeeze()
            y_true.extend(y.numpy())
            y_pred.extend(preds.cpu().numpy())
    auc = roc_auc_score(y_true, y_pred)
    acc = accuracy_score(y_true, np.round(y_pred))
    return auc, acc


In [7]:
print("✅ Training on:", device)
print(next(model.parameters()).device)  # Kiểm tra model hiện ở GPU hay CPU

✅ Training on: cuda
cuda:0


In [8]:
# Cell 7: Train loop
EPOCHS = 10
for epoch in range(EPOCHS):
    loss = train_model(model, train_loader, criterion, optimizer, device)
    auc, acc = evaluate_model(model, test_loader, device)
    print(f"📍 Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | AUC: {auc:.4f} | ACC: {acc:.4f}")


📍 Epoch 1/10 | Loss: 0.4632 | AUC: 0.8581 | ACC: 0.7820
📍 Epoch 2/10 | Loss: 0.4238 | AUC: 0.8653 | ACC: 0.7892
📍 Epoch 3/10 | Loss: 0.4032 | AUC: 0.8672 | ACC: 0.7901
📍 Epoch 4/10 | Loss: 0.3904 | AUC: 0.8668 | ACC: 0.7882
📍 Epoch 5/10 | Loss: 0.3818 | AUC: 0.8659 | ACC: 0.7875
📍 Epoch 6/10 | Loss: 0.3753 | AUC: 0.8640 | ACC: 0.7864
📍 Epoch 7/10 | Loss: 0.3697 | AUC: 0.8624 | ACC: 0.7842
📍 Epoch 8/10 | Loss: 0.3646 | AUC: 0.8585 | ACC: 0.7800
📍 Epoch 9/10 | Loss: 0.3598 | AUC: 0.8555 | ACC: 0.7766
📍 Epoch 10/10 | Loss: 0.3551 | AUC: 0.8539 | ACC: 0.7752


In [9]:
# Cell 8: Test prediction
sample = test_df.iloc[0:5]
x_cat = torch.tensor(sample[categorical_cols].values, dtype=torch.long).to(device)
x_num = torch.tensor(sample[numeric_cols].values, dtype=torch.float32).to(device)
preds = model(x_cat, x_num).detach().cpu().numpy()
print("✅ Predicted click probabilities:", preds.flatten())

✅ Predicted click probabilities: [9.5074463e-01 3.1171668e-02 7.0747620e-01 9.3888948e-03 7.0765651e-07]


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_metrics(model, data_loader, device, threshold=0.5):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x_cat, x_num, y in data_loader:
            x_cat, x_num = x_cat.to(device), x_num.to(device)
            preds = model(x_cat, x_num).squeeze()
            y_true.extend(y.numpy())
            y_pred.extend((preds.cpu().numpy() > threshold).astype(int))
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return acc, prec, rec, f1

In [11]:
acc, prec, rec, f1 = evaluate_metrics(model, test_loader, device)
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 0.7752
Precision: 0.7195
Recall: 0.6973
F1 Score: 0.7082
